# Competition PDF WebSec Training on Colab

This notebook replaces the old experimental notebook.

It does five things cleanly:
- clones the latest repo
- uses `DATA.pdf` when available, or the prebuilt JSONL fallback when it is not
- builds a merged training set from the repo baseline + competition PDF cases
- fine-tunes `Qwen/Qwen2.5-Coder-1.5B-Instruct` with T4-safe QLoRA settings
- runs eval, smoke tests, and downloads the adapters

Recommended runtime: **T4 GPU** or better.

## 1. Check GPU

If this cell does not show an NVIDIA GPU, switch Colab to:
`Runtime -> Change runtime type -> T4 GPU`.

In [ ]:
!nvidia-smi

## 2. Clone the Repo

This notebook assumes the training code lives in the GitHub repo.

In [ ]:
%cd /content
!rm -rf imkc
!git clone https://github.com/dakshdoesdev/imkc.git
%cd /content/imkc

!git rev-parse --short HEAD
!pwd

## 3. Install Dependencies

`poppler-utils` gives us `pdftotext`, which the PDF builder can use.
The builder also supports `pypdf` as a fallback.

In [ ]:
!apt-get -qq update
!apt-get -qq install -y poppler-utils
!pip install -q -r training_requirements.txt pypdf

print('Dependencies installed')

## 4. Resolve the Competition Data

Priority order:
1. Use `COMPETEION INSTRCUTONS + MICELL/DATA.pdf` if it exists in the repo
2. Otherwise use the tracked derived dataset `training/competition_pdf_train.jsonl`
3. Otherwise upload `DATA.pdf` manually and rebuild the dataset

In [ ]:
from pathlib import Path
import os

repo_root = Path('/content/imkc')
competition_dir = repo_root / 'COMPETEION INSTRCUTONS + MICELL'
repo_pdf = competition_dir / 'DATA.pdf'
builder_script = repo_root / 'training' / 'build_competition_pdf_dataset.py'
pdf_dataset = repo_root / 'training' / 'competition_pdf_train.jsonl'

print('Repo PDF exists:', repo_pdf.exists())
print('Prebuilt PDF dataset exists:', pdf_dataset.exists())
print('Builder exists:', builder_script.exists())

In [ ]:
from pathlib import Path
import os

repo_root = Path('/content/imkc')
os.chdir(repo_root)

repo_pdf = repo_root / 'COMPETEION INSTRCUTONS + MICELL' / 'DATA.pdf'
pdf_dataset = repo_root / 'training' / 'competition_pdf_train.jsonl'
builder_script = repo_root / 'training' / 'build_competition_pdf_dataset.py'

if repo_pdf.exists():
    print(f'Using repo PDF: {repo_pdf}')
    !python {builder_script} --pdf {repo_pdf} --output {pdf_dataset}
elif pdf_dataset.exists():
    print(f'PDF not found in repo. Using tracked derived dataset: {pdf_dataset}')
else:
    from google.colab import files

    print('Upload DATA.pdf now')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('No file uploaded. Upload DATA.pdf and rerun this cell.')

    uploaded_name = next(iter(uploaded))
    uploaded_pdf = repo_root / uploaded_name
    print(f'Uploaded file: {uploaded_pdf}')
    !python {builder_script} --pdf {uploaded_pdf} --output {pdf_dataset}

print('Competition dataset ready at:', pdf_dataset)

## 5. Preview the Competition Dataset

In [ ]:
from pathlib import Path
import json
from collections import Counter

pdf_dataset = Path('/content/imkc/training/competition_pdf_train.jsonl')
family_counts = Counter()
action_counts = Counter()
vuln_counts = Counter()

with pdf_dataset.open(encoding='utf-8') as handle:
    rows = [json.loads(line) for line in handle if line.strip()]

for row in rows:
    family_counts[row['example_family']] += 1
    action = row['output']['action']
    action_counts[action['action_type']] += 1
    vuln_counts[action['vulnerability_type']] += 1

print('Competition rows:', len(rows))
print('Example families:', dict(family_counts))
print('Action counts:', dict(action_counts))
print('Vulnerability counts:', dict(vuln_counts))
print()
print('Sample task_id:', rows[0]['task_id'])
print('Sample instruction:', rows[0]['instruction'])
print('Sample output:', json.dumps(rows[0]['output'], indent=2))

## 6. Merge the Repo Baseline with the Competition Dataset

This keeps the original 100 baseline rows and adds the PDF-derived cases.

In [ ]:
from pathlib import Path
import json

repo_root = Path('/content/imkc')
base_dataset = repo_root / 'training' / 'websec_repair_train_100.jsonl'
pdf_dataset = repo_root / 'training' / 'competition_pdf_train.jsonl'
merged_dataset = repo_root / 'training' / 'competition_merged_train.jsonl'

merged_count = 0
with merged_dataset.open('w', encoding='utf-8') as out:
    for dataset_path in (base_dataset, pdf_dataset):
        with dataset_path.open(encoding='utf-8') as handle:
            for line in handle:
                if line.strip():
                    out.write(line)
                    merged_count += 1

print('Base dataset:', sum(1 for _ in base_dataset.open(encoding='utf-8')))
print('Competition dataset:', sum(1 for _ in pdf_dataset.open(encoding='utf-8')))
print('Merged dataset:', merged_count)
print('Merged path:', merged_dataset)

In [ ]:
from training.data_loader import load_jsonl_dataset, summarize_dataset

merged_dataset = '/content/imkc/training/competition_merged_train.jsonl'
summary = summarize_dataset(load_jsonl_dataset(merged_dataset))
summary

## 7. Build the External WebSec Corpus

If you have raw PrimeVul, Devign, Juliet, OWASP Benchmark, CVEfixes, or other exports,
you can pass them to the builder later. This cell at least creates the built-in synthetic patterns.

In [ ]:
!python training/build_external_websec_corpus.py --output training/external_websec_corpus.jsonl

## 8. Training Configuration

These defaults are safer for a T4 than the old notebook:
- `batch_size = 1`
- `gradient_accumulation_steps = 16`
- `PYTORCH_ALLOC_CONF=expandable_segments:True`

`INCLUDE_TRAJECTORIES` keeps the environment workflow examples (`inspect -> classify -> apply_patch -> verify -> submit`).

In [ ]:
OUTPUT_DIR = './qwen-websec-competition-finetuned'
MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 16
INCLUDE_TRAJECTORIES = True

print({
    'OUTPUT_DIR': OUTPUT_DIR,
    'MODEL_ID': MODEL_ID,
    'EPOCHS': EPOCHS,
    'LEARNING_RATE': LEARNING_RATE,
    'BATCH_SIZE': BATCH_SIZE,
    'GRAD_ACCUM_STEPS': GRAD_ACCUM_STEPS,
    'INCLUDE_TRAJECTORIES': INCLUDE_TRAJECTORIES,
})

In [ ]:
import os
import sys
import subprocess
import torch

if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU detected. Switch Colab to a T4 GPU runtime before training.')

os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

cmd = [
    sys.executable,
    'training/qlora_train.py',
    '--output-dir', OUTPUT_DIR,
    '--model-id', MODEL_ID,
    '--jsonl-dataset', 'training/competition_merged_train.jsonl',
    '--research-doc', 'training/research_websec_review_playbook.md',
    '--external-websec-jsonl', 'training/external_websec_corpus.jsonl',
    '--no-include-swe-bench',
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--gradient-accumulation-steps', str(GRAD_ACCUM_STEPS),
    '--learning-rate', str(LEARNING_RATE),
]

if INCLUDE_TRAJECTORIES:
    cmd.extend(['--trajectory-logs', 'trajectory_logs.json'])

print('Running:')
print(' '.join(cmd))
print()
subprocess.run(cmd, check=True)

## 9. Run Evaluation

In [ ]:
!python eval/websec_eval.py   --model-dir {OUTPUT_DIR}   --output-json ./artifacts/competition_pdf_eval_report.json

In [ ]:
import json
from pathlib import Path

report_path = Path('./artifacts/competition_pdf_eval_report.json')
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report, indent=2)[:4000])
else:
    print('Eval report not found:', report_path)

## 10. Smoke-Test the Fine-Tuned Model

These checks target the granular labels you were explicitly looking for.

In [ ]:
from training.inference import WebSecRepairAgent
import json

agent = WebSecRepairAgent(OUTPUT_DIR)

tests = [
    {
        'name': 'Hardcoded credentials',
        'task_id': 'hardcoded_creds_test',
        'instruction': 'Identify hardcoded administrative credentials',
        'code_snippet': 'if username == "admin" and password == "P@ssword123": login_user()',
        'scanner_hint': 'Hardcoded credentials found in authentication logic',
    },
    {
        'name': 'Path traversal',
        'task_id': 'path_traversal_test',
        'instruction': 'Validate file path to prevent directory traversal',
        'code_snippet': 'file_path = "/var/www/static/" + request.args.get("file")
with open(file_path, "rb") as f: return f.read()',
        'scanner_hint': 'Unsanitized file path construction detected',
    },
    {
        'name': 'Command injection',
        'task_id': 'command_injection_test',
        'instruction': 'Sanitize input before passing to a system shell',
        'code_snippet': 'os.system(f"ls {user_supplied_path}")',
        'scanner_hint': 'Shell metacharacters in user input could allow arbitrary command execution',
    },
    {
        'name': 'SQL injection',
        'task_id': 'sqli_test',
        'instruction': 'Fix SQL injection in login form',
        'code_snippet': 'query = f"SELECT * FROM users WHERE id={uid}"',
        'scanner_hint': 'SQL injection detected in query construction',
    },
]

for test in tests:
    print('=' * 80)
    print(test['name'])
    result = agent.get_action(
        task_id=test['task_id'],
        instruction=test['instruction'],
        code_snippet=test['code_snippet'],
        scanner_hint=test['scanner_hint'],
    )
    print(json.dumps(result, indent=2))

## 11. Download the Trained Adapters

This zips the full output directory so you can download the adapters, tokenizer, and metadata.

In [ ]:
from google.colab import files
from pathlib import Path

output_path = Path(OUTPUT_DIR)
if not output_path.exists():
    raise RuntimeError(f'Model output directory not found: {output_path}')

archive_name = 'qwen_websec_competition_finetuned.zip'
!rm -f {archive_name}
!zip -rq {archive_name} {OUTPUT_DIR}
files.download(archive_name)
print('Download started:', archive_name)